# Notebook 15: Cross-Source Diagnostic Checks and Device Mismatch Analysis

LRNG: National (India) | CC: Human Values | SDG: 3 | Data: Time series | Thrust: Intelligent Edge Technologies | Screening aid only

Objectives:
1. Run Seizure_Conv1D_BiLSTM.tflite on PADS PD, ET, and Healthy tremor-task windows resampled from 100 Hz to 16 Hz through the config pipeline.
2. Report the fraction of windows and subjects flagged seizure-like per class.
3. Run Strided_Conv1D_BiLSTM.tflite on UEA seizure and normal series upsampled from 16 Hz to 100 Hz (with zero-padded gyroscope channels) and report predicted class counts.
4. Explicitly label both checks as cross-source diagnostic stress tests confounded by device mismatch and missing modalities, not validation.
5. Save consolidated metrics to results/metrics/cross_check.csv.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy.signal import butter, lfilter, resample_poly

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
np.random.seed(42)
tf.random.set_seed(42)


## 1. Cross-Check 1: Evaluating Seizure Model on PADS Movement Disorder Data
Testing Seizure_Conv1D_BiLSTM.tflite on PADS 100 Hz accelerometer data resampled to 16 Hz through the complete preprocessing pipeline.

In [2]:
pads_data = np.load(os.path.join("..", "results", "pads_primary_100hz.npz"))["windows"]
pads_meta = pd.read_csv(os.path.join("..", "results", "pads_primary_100hz_metadata.csv"))
accel_100hz = pads_data[:, :, :3]

print("Resampling 25,500 PADS windows from 100 Hz to 16 Hz...")
accel_16hz = resample_poly(accel_100hz, 4, 25, axis=1)

b_sz, a_sz = butter(4, [3.0, 7.5], btype="bandpass", fs=16.0)
filtered_16hz = lfilter(b_sz, a_sz, accel_16hz, axis=1)
demeaned_16hz = (filtered_16hz - np.mean(filtered_16hz, axis=1, keepdims=True)).astype(np.float32)

uea_data = np.load(os.path.join("..", "results", "uea_within_source_16hz.npz"))["windows"]
uea_meta = pd.read_csv(os.path.join("..", "results", "uea_within_source_metadata.csv"))
train_mask = (uea_meta["split"] == "train").values
ch_std_uea = (uea_data[train_mask] - uea_data[train_mask].mean(axis=1, keepdims=True)).std(axis=(0, 1))
ch_std_uea[ch_std_uea == 0] = 1.0
norm_16hz = (demeaned_16hz / ch_std_uea).astype(np.float32)

tflite_seizure_path = os.path.join("..", "results", "models", "Seizure_Conv1D_BiLSTM.tflite")
interp_sz = tf.lite.Interpreter(model_path=tflite_seizure_path)
in_idx = interp_sz.get_input_details()[0]["index"]
out_idx = interp_sz.get_output_details()[0]["index"]

bs = 1000
interp_sz.resize_tensor_input(in_idx, [bs, 48, 3])
interp_sz.allocate_tensors()

probs_all = []
for i in range(0, len(norm_16hz), bs):
    chunk = norm_16hz[i:i+bs]
    if len(chunk) < bs:
        pad = np.zeros((bs - len(chunk), 48, 3), dtype=np.float32)
        chunk_padded = np.vstack([chunk, pad])
        interp_sz.set_tensor(in_idx, chunk_padded)
        interp_sz.invoke()
        p = interp_sz.get_tensor(out_idx)[:len(chunk)]
    else:
        interp_sz.set_tensor(in_idx, chunk)
        interp_sz.invoke()
        p = interp_sz.get_tensor(out_idx)
    probs_all.append(p[:, 1])

probs_seizure = np.concatenate(probs_all)
pads_meta["seizure_prob"] = probs_seizure
pads_meta["flagged_seizure"] = (probs_seizure >= 0.5).astype(int)

cc1_records = []
print("=== Cross-Check 1: Seizure Model on PADS Cohorts (Diagnostic Check, Not Validation) ===")
for label in ["Normal", "Parkinsons-like", "ET-like"]:
    sub_df = pads_meta[pads_meta["cohort_label"] == label]
    win_flagged = int(sub_df["flagged_seizure"].sum())
    win_total = int(len(sub_df))
    win_frac = win_flagged / win_total
    
    sub_means = sub_df.groupby("subject_id")["seizure_prob"].mean()
    sub_flagged = int((sub_means >= 0.5).sum())
    sub_total = int(len(sub_means))
    sub_frac = sub_flagged / sub_total
    
    print(f"Cohort {label}:")
    print(f"  Windows Flagged Seizure-like: {win_flagged} of {win_total} ({win_frac:.2%})")
    print(f"  Subjects Flagged Seizure-like: {sub_flagged} of {sub_total} ({sub_frac:.2%})")
    
    cc1_records.append({
        "Test": "Seizure_Model_on_PADS",
        "Cohort": label,
        "Total_Windows": win_total,
        "Flagged_Windows": win_flagged,
        "Window_Flagged_Fraction": win_frac,
        "Total_Subjects": sub_total,
        "Flagged_Subjects": sub_flagged,
        "Subject_Flagged_Fraction": sub_frac
    })


Resampling 25,500 PADS windows from 100 Hz to 16 Hz...


=== Cross-Check 1: Seizure Model on PADS Cohorts (Diagnostic Check, Not Validation) ===
Cohort Normal:
  Windows Flagged Seizure-like: 0 of 5372 (0.00%)
  Subjects Flagged Seizure-like: 0 of 79 (0.00%)
Cohort Parkinsons-like:
  Windows Flagged Seizure-like: 232 of 18224 (1.27%)
  Subjects Flagged Seizure-like: 1 of 268 (0.37%)
Cohort ET-like:
  Windows Flagged Seizure-like: 66 of 1904 (3.47%)
  Subjects Flagged Seizure-like: 0 of 28 (0.00%)


## 2. Cross-Check 2: Evaluating Tremor Model on Upsampled UEA Seizure and Normal Series
Testing Strided_Conv1D_BiLSTM.tflite on UEA Epilepsy series upsampled from 16 Hz to 100 Hz with zero-padded gyroscope channels (only for this cross-source test).

In [3]:
accel_100hz_uea = resample_poly(uea_data, 25, 4, axis=1)
zero_gyro = np.zeros((len(uea_data), 300, 3), dtype=np.float32)
full_6ch_100hz = np.concatenate([accel_100hz_uea, zero_gyro], axis=2).astype(np.float32)

pads_raw = np.load(os.path.join("..", "results", "pads_primary_100hz.npz"))["windows"]
ch_std_pads = (pads_raw - pads_raw.mean(axis=1, keepdims=True)).std(axis=(0, 1))
ch_std_pads[ch_std_pads == 0] = 1.0

demeaned_6ch = full_6ch_100hz - np.mean(full_6ch_100hz, axis=1, keepdims=True)
norm_6ch = (demeaned_6ch / ch_std_pads).astype(np.float32)

tflite_tremor_path = os.path.join("..", "results", "models", "Strided_Conv1D_BiLSTM.tflite")
interp_tr = tf.lite.Interpreter(model_path=tflite_tremor_path)
in_idx_tr = interp_tr.get_input_details()[0]["index"]
out_idx_tr = interp_tr.get_output_details()[0]["index"]

interp_tr.resize_tensor_input(in_idx_tr, [bs, 300, 6])
interp_tr.allocate_tensors()

probs_all_tr = []
for i in range(0, len(norm_6ch), bs):
    chunk = norm_6ch[i:i+bs]
    if len(chunk) < bs:
        pad = np.zeros((bs - len(chunk), 300, 6), dtype=np.float32)
        chunk_padded = np.vstack([chunk, pad])
        interp_tr.set_tensor(in_idx_tr, chunk_padded)
        interp_tr.invoke()
        p = interp_tr.get_tensor(out_idx_tr)[:len(chunk)]
    else:
        interp_tr.set_tensor(in_idx_tr, chunk)
        interp_tr.invoke()
        p = interp_tr.get_tensor(out_idx_tr)
    probs_all_tr.append(p)

probs_tremor = np.vstack(probs_all_tr)
pred_classes = np.argmax(probs_tremor, axis=1)

uea_meta["pred_class"] = pred_classes
class_names = ["Normal", "Parkinsons-like", "ET-like"]
uea_meta["pred_label"] = [class_names[c] for c in pred_classes]

print("=== Cross-Check 2: Tremor Model on Upsampled UEA (Diagnostic Check, Not Validation) ===")
print("Window-Level Class Counts:")
cc2_records = []
for cohort in ["Normal", "Seizure-like (simulated)"]:
    sub = uea_meta[uea_meta["cohort_label"] == cohort]
    counts = sub["pred_label"].value_counts().to_dict()
    norm_c = counts.get("Normal", 0)
    pd_c = counts.get("Parkinsons-like", 0)
    et_c = counts.get("ET-like", 0)
    print(f"True {cohort} ({len(sub)} windows): Normal={norm_c}, PD={pd_c}, ET={et_c}")

series_preds = []
for sid in uea_meta["series_id"].unique():
    s_rows = uea_meta[uea_meta["series_id"] == sid]
    s_true = s_rows["cohort_label"].iloc[0]
    mean_p = np.mean(probs_tremor[s_rows.index], axis=0)
    p_class = class_names[np.argmax(mean_p)]
    series_preds.append({"series_id": sid, "true_cohort": s_true, "pred_class": p_class})

df_series = pd.DataFrame(series_preds)
print("\nSeries-Level Class Counts (Averaged Window Probabilities):")
for cohort in ["Normal", "Seizure-like (simulated)"]:
    sub = df_series[df_series["true_cohort"] == cohort]
    counts = sub["pred_class"].value_counts().to_dict()
    norm_c = counts.get("Normal", 0)
    pd_c = counts.get("Parkinsons-like", 0)
    et_c = counts.get("ET-like", 0)
    print(f"True {cohort} ({len(sub)} series): Normal={norm_c}, PD={pd_c}, ET={et_c}")
    
    cc2_records.append({
        "Test": "Tremor_Model_on_Upsampled_UEA",
        "Cohort": cohort,
        "Total_Units": len(sub),
        "Pred_Normal": norm_c,
        "Pred_Parkinsons": pd_c,
        "Pred_ET": et_c
    })


=== Cross-Check 2: Tremor Model on Upsampled UEA (Diagnostic Check, Not Validation) ===
Window-Level Class Counts:
True Normal (1449 windows): Normal=0, PD=885, ET=564
True Seizure-like (simulated) (476 windows): Normal=0, PD=357, ET=119

Series-Level Class Counts (Averaged Window Probabilities):
True Normal (207 series): Normal=0, PD=128, ET=79
True Seizure-like (simulated) (68 series): Normal=0, PD=52, ET=16


## 3. Save Consolidated Cross-Check Metrics
Writing results to results/metrics/cross_check.csv.

In [4]:
df_cc1 = pd.DataFrame(cc1_records)
df_cc2 = pd.DataFrame(cc2_records)

df_all = pd.concat([df_cc1, df_cc2], ignore_index=True)
out_csv = os.path.join("..", "results", "metrics", "cross_check.csv")
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
df_all.to_csv(out_csv, index=False)
print(f"Saved cross-check metrics to {out_csv}")
print(df_all.to_string())


Saved cross-check metrics to ..\results\metrics\cross_check.csv
                            Test                    Cohort  Total_Windows  Flagged_Windows  Window_Flagged_Fraction  Total_Subjects  Flagged_Subjects  Subject_Flagged_Fraction  Total_Units  Pred_Normal  Pred_Parkinsons  Pred_ET
0          Seizure_Model_on_PADS                    Normal         5372.0              0.0                 0.000000            79.0               0.0                  0.000000          NaN          NaN              NaN      NaN
1          Seizure_Model_on_PADS           Parkinsons-like        18224.0            232.0                 0.012730           268.0               1.0                  0.003731          NaN          NaN              NaN      NaN
2          Seizure_Model_on_PADS                   ET-like         1904.0             66.0                 0.034664            28.0               0.0                  0.000000          NaN          NaN              NaN      NaN
3  Tremor_Model_on_Upsam